In [ ]:
import os
import numpy as np
import tkinter as tk
from tkinter import filedialog, messagebox
from PIL import Image, ImageTk
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model

# --- CONFIGURATION ---
MODEL_PATH = 'MyModel.keras'
IMG_SIZE = (224, 224)
CLASS_NAMES = ['Cat', 'Dog']

# Load model once at startup
try:
    model = load_model(MODEL_PATH)
except Exception as e:
    print(f"Error loading model: {e}")
    model = None

class PredictionApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Cat-Dog Classifier")
        self.root.geometry("450x600")
        self.root.configure(bg="#f0f0f0")

        # UI Elements
        self.label_title = tk.Label(root, text="Pet Predictor", font=("Arial", 18, "bold"), bg="#f0f0f0")
        self.label_title.pack(pady=10)

        self.canvas = tk.Canvas(root, width=300, height=300, bg="white", highlightthickness=1)
        self.canvas.pack(pady=10)

        self.btn_load = tk.Button(root, text="Load Image", command=self.load_and_predict, 
                                  font=("Arial", 12), bg="#4CAF50", fg="white", padx=20)
        self.btn_load.pack(pady=10)

        self.result_text = tk.StringVar(value="Result: Waiting for image...")
        self.label_result = tk.Label(root, textvariable=self.result_text, font=("Arial", 14), bg="#f0f0f0")
        self.label_result.pack(pady=5)

        self.conf_text = tk.StringVar(value="")
        self.label_conf = tk.Label(root, textvariable=self.conf_text, font=("Arial", 12, "italic"), bg="#f0f0f0")
        self.label_conf.pack()

    def load_and_predict(self):
        file_path = filedialog.askopenfilename(filetypes=[("Image files", "*.jpg *.jpeg *.png")])
        
        if not file_path:
            return

        # 1. Display Preview
        preview_img = Image.open(file_path)
        preview_img.thumbnail((300, 300)) # Resize for GUI display
        self.tk_img = ImageTk.PhotoImage(preview_img)
        self.canvas.create_image(150, 150, image=self.tk_img)

        # 2. Preprocess for Model
        try:
            test_img = image.load_img(file_path, target_size=IMG_SIZE)
            img_array = image.img_to_array(test_img)
            img_batch = np.expand_dims(img_array, axis=0)

            # 3. Predict
            if model:
                preds = model.predict(img_batch, verbose=0)
                idx = np.argmax(preds[0])
                conf = np.max(preds[0]) * 100

                # Update GUI
                self.result_text.set(f"Result: {CLASS_NAMES[idx]}")
                self.conf_text.set(f"Confidence: {conf:.2f}%")
                
                # Highlight if in your 50-70% range
                if 50 <= conf <= 70:
                    self.label_result.config(fg="orange")
                else:
                    self.label_result.config(fg="black")
            else:
                messagebox.showerror("Error", "Model not loaded!")
        except Exception as e:
            messagebox.showerror("Error", f"Prediction failed: {e}")

# Run the App
if __name__ == "__main__":
    root = tk.Tk()
    app = PredictionApp(root)
    root.mainloop()